In [ ]:
import sys, os, shutil

# Đường dẫn theo CLAUDE.md (chạy trên Kaggle):
#   UTILITY_PATH = "/kaggle/input/notebooks/laithetrung/waft-utility-script"
#   CODE_SRC     = "/kaggle/input/notebooks/laithetrung/waft-repo/JeDepth"
#   DATASET_SRC  = "/kaggle/input/datasets/laithetrung/stereo-smallbaseline"
UTILITY_PATH = "/kaggle/input/notebooks/laithetrung/waft-utility-script"
CODE_SRC     = "/kaggle/input/notebooks/laithetrung/waft-repo/JeDepth"
DATASET_SRC  = "/kaggle/input/datasets/laithetrung/stereo-smallbaseline"

# Fallbacks cho convention `/kaggle/input/<slug>/...`
for p in ["/kaggle/input/waft-utility/kaggle/working", "/kaggle/input/waft-utility"]:
    if os.path.exists(p) and not os.path.exists(UTILITY_PATH):
        UTILITY_PATH = p
        break
for p in ["/kaggle/input/jedepth-code/JeDepth", "/kaggle/input/jedepth-code", "/kaggle/input/waft-repo"]:
    if os.path.exists(p) and not os.path.exists(CODE_SRC):
        CODE_SRC = p
        break
for p in ["/kaggle/input/stereo-smallbaseline"]:
    if os.path.exists(p) and not os.path.exists(DATASET_SRC):
        DATASET_SRC = p
        break

# Copy source code ra /kaggle/working để có thể ghi (ckpts, tb logs, etc.)
REPO_PATH = "/kaggle/working/JeDepth"
if not os.path.exists(REPO_PATH):
    print(f"Copying source: {CODE_SRC} → {REPO_PATH}")
    shutil.copytree(CODE_SRC, REPO_PATH)
print("Source size:", sum(os.path.getsize(os.path.join(d,f)) for d,_,fs in os.walk(REPO_PATH) for f in fs)//1024, "KB")

if os.path.exists(UTILITY_PATH):
    sys.path.insert(0, UTILITY_PATH)
sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("UTILITY_PATH:", UTILITY_PATH, os.path.exists(UTILITY_PATH))
print("REPO_PATH   :", REPO_PATH)
print("DATASET_SRC :", DATASET_SRC, os.path.exists(DATASET_SRC))

In [ ]:
!ls /kaggle/working/JeDepth

In [ ]:
# ── Link the dataset to the path expected by the CSVs ──
DATASET_TARGET_ROOT = "/kaggle/working/data"
os.makedirs(DATASET_TARGET_ROOT, exist_ok=True)
target = os.path.join(DATASET_TARGET_ROOT, "stereo-smallbaseline")
if not os.path.exists(target):
    os.symlink(DATASET_SRC, target)
print("Dataset linked at", target)
print("Files:", os.listdir(target)[:5])

In [ ]:
# ── Launch training. Working dir = /kaggle/working/JeDepth → ckpts tự động nằm trong working ──
import subprocess
os.environ["PYTHONPATH"] = f"{REPO_PATH}:{UTILITY_PATH}:" + os.environ.get("PYTHONPATH", "")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WAFT_NO_PRETRAINED"] = "1"

INIT_CKPT_CANDIDATES = [
    "/kaggle/input/notebooks/laithetrung/waft-repo/init_ckpt.pth",
    "/kaggle/input/waft-repo/init_ckpt.pth",
]
INIT_CKPT = next((p for p in INIT_CKPT_CANDIDATES if os.path.exists(p)), None)
print("INIT_CKPT:", INIT_CKPT)

# Trỏ ROOT thẳng vào dataset folder; _resolve_path sẽ tự strip prefix
# (cả "stereo-smallbaseline/..." lẫn "processed_data/...") để tìm "left/...".
DATA_ROOT = target  # /kaggle/working/data/stereo-smallbaseline → symlink tới DATASET_SRC

cmd = [
    "python", "main.py",
    "--config-file", "configs/Real/stereo_smallbaseline.yaml",
    "--num-gpus", "1",
    "--seed", "42",
]
if INIT_CKPT is not None:
    cmd += ["--ckpt", INIT_CKPT]
cmd += [
    "DATASETS.ROOT", DATA_ROOT,
    "DATASETS.TRAIN_CSV", os.path.join(target, "train.csv"),
    "DATASETS.VAL_CSV",   os.path.join(target, "val.csv"),
    "TEST.TEST_IMAGES_DIR", os.path.join(REPO_PATH, "test_images"),
    "SOLVER.IMS_PER_BATCH", "8",
    "SOLVER.BASE_LR", "0.0002",
    "SOLVER.MAX_EPOCH", "60",
    "TEST.EVAL_EPOCH_PERIOD", "5",
    "DATASETS.CROP_SIZE", "[544,960]",
]
print("Launching:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Done. Checkpoints + TB logs ở", os.path.join(REPO_PATH, "ckpts"))